In [ ]:
import fasttext.util
import numpy as np
import pandas as pd
import re
import os
from scipy import stats
from scipy.stats import skew, kurtosis
from collections import Counter
from nltk.corpus import stopwords
import nltk
import spacy
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns

ft = fasttext.load_model(r"Data/Models/cc.de.300.bin") #fasttext model path
print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
#folder path
folder_path = r"Data/Corpus"

nlp = spacy.load("de_core_news_md")

nltk.download('stopwords', quiet=True)
german_stop_words = set(stopwords.words('german'))

key_distinction_1 = ['Natur']
key_distinction_2 = ['Kultur']

#key_distinction_1 = ['forms_of_representation_for_key_distinction_1']
#key_distinction_2 = ['forms_of_representation_for_key_distinction_2']

"""
#Example

#for adjectives
key_distinction_1 = ['naturverbunden']
key_distinction_2 = ['zivilisiert']

#average of synonyms
key_distinction_1 = ['naturverbunden', 'naturbelassen', 'rein', 'unverfälscht', 'lebendig', 'natürlich']
key_distinction_2 = ['zivilisiert', 'kultiviert', 'künstlich', 'artifiziell', 'hochentwickelt', 'gebildet', 'gekünstelt']

#nouns
key_distinction_1 = ['Natur']
key_distinction_2 = ['Kultur']
"""

print("Checking OOV words for the selected synonyms:")
for w in key_distinction_1 + key_distinction_2:
    if ft.get_word_id(w) == -1:
        print(f"'{w}' is OOV!")

#find axis vector and normalize it
def get_axis_vector():
    key_distinction_1_vecs = [ft.get_word_vector(w) for w in key_distinction_1]
    key_distinction_2_vecs = [ft.get_word_vector(w) for w in key_distinction_2]
    key_distinction_1_mean = np.mean(key_distinction_1_vecs, axis=0)
    key_distinction_2_mean = np.mean(key_distinction_2_vecs, axis=0)
    axis = key_distinction_2_mean - key_distinction_1_mean
    return axis / np.linalg.norm(axis)

#find cosine similarities
def cosine_to_axis(word, axis_vec):
    v = ft.get_word_vector(word)
    norm = np.linalg.norm(v)
    if norm == 0:
        return 0.0
    v_norm = v / norm
    return float(np.dot(v_norm, axis_vec))

#handle empty lists to avoid errors
def summary_stats(scores):
    if not scores: 
        return {k: 0.0 for k in ["min", "max", "diff", "mean", "median", "std", "skew", "kurt"]}
        
    scores = np.array(scores)
    return {
        "min":    float(np.min(scores)),
        "max":    float(np.max(scores)),
        "diff":   float(np.max(scores)-np.min(scores)),
        "mean":   float(np.mean(scores)),
        "median": float(np.median(scores)),
        "std":    float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "skew":   float(stats.skew(scores, bias=False)) if len(scores) > 1 else 0.0,
        "kurt":   float(stats.kurtosis(scores, fisher=True, bias=False)) if len(scores) > 1 else 0.0,
    }

def calculate_file_bias(text, axis_vec, seed_words):
    words = re.sub(r"[^\w\s]", "", text.lower()).split()
    
    unique_text_words = set(words)
    
    #Keeps only words that are stopwords
    neutral_candidates = unique_text_words.intersection(german_stop_words)
    
    scores = []
    for w in neutral_candidates:
        # Checks for seed words
        if w in seed_words: continue
        
        #Checks if the word exists in the FastText model
        if ft.get_word_id(w) != -1:
            scores.append(cosine_to_axis(w, axis_vec))

    if not scores: return 0.0
    return np.mean(scores)

#function for the pipline
def process_text_pipeline(text):
    #Runs SpaCy on RAW text, Filters NOUN/ADJ, Lowercases and cleans punctuation, Removes Stopwords, Checks if word exists in FastText Vocab
    doc = nlp(text)
    valid_words = []
    total_count = 0
    oov = 0
    
    for token in doc:
        #POS Tagging
        if token.pos_ in {"NOUN", "ADJ"}:
            #Lowercase & Clean
            w_clean = re.sub(r"[^\w\s]", "", token.text.lower())
            #Check Stopwords & Empty strings
            if w_clean and w_clean not in german_stop_words:
                total_count +=1 #keeps all the words
                if ft.get_word_id(w_clean) != -1: #Check FastText Vocabulary
                    valid_words.append(w_clean)
                else:
                    oov +=1

    if total_count > 0:
        pct = (oov / total_count) * 100
        print(f"Vocabulary Info: {total_count} total words, {oov} OOV ({pct:.1f}% dropped)")
    else:
        print(f"Vocabulary Info: No candidates found.")
    return valid_words

axis_vec = get_axis_vector()

#stats_unique = {} 
stats_freq   = {} 
u_stats_freq   = {} 

#For the file list
text_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".txt")]
text_files = sorted(text_files) 

all_seed_words = set(key_distinction_1 + key_distinction_2)

violin_data = [] 

for fname in text_files:
    file_path = os.path.join(folder_path, fname)

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            text_document = f.read()
    except Exception as e:
        print(f"Error reading {fname}: {e}")
        continue

    file_bias = calculate_file_bias(text_document, axis_vec, all_seed_words)
    print(f"Processing: {fname}")
    print(f"Found Bias: {file_bias:.4f}")

    content_words = process_text_pipeline(text_document)

    if not content_words:
        print(f"Warning: No valid content words found in {fname}")
        continue

    #Calculate Unique words
    unique_words = list(set(content_words))
    unique_scores_map = {}
    
    for w in unique_words:
        raw = cosine_to_axis(w, axis_vec)
        corrected = raw - file_bias
        unique_scores_map[w] = corrected

    full_doc_scores = [unique_scores_map[w] for w in content_words]

    u_stats_freq[fname] = summary_stats(full_doc_scores)
    u_f_st = u_stats_freq[fname]

    print("Statistical Measures:")
    print(f"  min    : {u_f_st['min']:.4f}")
    print(f"  max    : {u_f_st['max']:.4f}")
    print(f"  diff   : {u_f_st['diff']:.4f}")
    print(f"  mean   : {u_f_st['mean']:.4f}")
    print(f"  median : {u_f_st['median']:.4f}")
    print(f"  std    : {u_f_st['std']:.4f}")
    print(f"  skew   : {u_f_st['skew']:.4f}")
    print(f"  kurt   : {u_f_st['kurt']:.4f}")

    filtered_entries = list(unique_scores_map.keys())

    filtered_doc_scores = full_doc_scores
    
    stats_freq[fname] = summary_stats(filtered_doc_scores)

    #score for violin plot
    for score in filtered_doc_scores:
        violin_data.append({
            "File": fname,
            "Score": score
        })

    #Bubble Plot
    freqs = Counter(content_words)

    plot_words = []
    plot_scores = []
    plot_counts = []

    for w in filtered_entries:
        plot_words.append(w)
        plot_scores.append(unique_scores_map[w]) 
        plot_counts.append(freqs[w])             

       #Apply random Jitters
    ys = np.random.uniform(-0.1, 0.1, size=len(plot_scores))

    sizes = [c * 30 for c in plot_counts]

    plt.figure(figsize=(12, 8))
    plt.scatter(plot_scores, ys, s=sizes, alpha=0.5, c='skyblue', edgecolors='grey')

    #Add Labels
    for x, y, lab, count in zip(plot_scores, ys, plot_words, plot_counts):
            plt.text(x, y, lab, fontsize=9, alpha=0.8)

    plt.axvline(0, color="red", linestyle="--", linewidth=1, label="Semantic Axis")
    
    plt.yticks([]) 
    plt.xlabel("<- key_distinction_1   VS   key_distinction_2 ->")
    plt.title(f"Semantic Distribution")
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [8]:
def plot_comparative_violin(data_list, x_filenames):
    if not data_list:
        return

    df = pd.DataFrame(data_list)

    # Force ordering: x_filenames first, then all others
    all_files = df["File"].unique().tolist()
    #Literary Narratives

    ordered_files = x_filenames + [f for f in all_files if f not in x_filenames]

    plt.figure(figsize=(6, 7))

    ax = sns.violinplot(
        data=df,
        x="File",
        y="Score",
        hue="File",
        order=ordered_files,   
        palette="RdBu",
        #palette="BrBG",
        inner="quartile",
        legend=False
    )

    plt.axhline(0, color='black', linestyle='-', linewidth=1.5)
    plt.ylim(-0.4, 0.4)

    labels = [tick.get_text().replace(".txt", "") for tick in ax.get_xticklabels()]
    ax.set_xticklabels(labels, rotation=45, ha='right')

    plt.ylabel("<- key_distinction_1al  |  Cultural ->", fontsize=12)
    plt.xlabel("Literary Narratives", fontsize=12)
    plt.tight_layout()

    plt.savefig("violin_x_marked_WT.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
x_filenames = ["file_names_goes_here"]
plot_comparative_violin(violin_data, x_filenames)